# 01 — Smoke train: does the whole training path run?

`smoke=True` is the project's own small-sample switch, not something invented
here — `bnhgq2.train.train` reads it and caps the run to **3 epochs over 2
`.h5` files** with no warmup. It exercises the same code the 6-hour cluster
jobs run, in a couple of minutes.

**This produces no result worth quoting.** The AUC printed below is from ~3
epochs on ~2% of the data. Nothing here goes near `RESEARCH.md`.

W&B is off in this pod (no API key is mounted), so nothing lands next to the
round-14 record.

### setup

In [ ]:
import os, json, time
from bnhgq2 import config as bncfg
from bnhgq2.train import train

CONFIG = os.path.join(os.environ["PYTHONPATH"], "configs", "r14-l1x3-n8-w1a8.json")
cfg = bncfg.load_config(CONFIG)
h = bncfg.cfg_hash(cfg)
out_dir = os.path.join(os.environ["BNHGQ2_OUT_ROOT"], f"smoke-{cfg['name']}-s1")

from bnhgq2 import wandb_util as wbu
print("config   :", cfg["name"], "| hash", h)
print("out_dir  :", out_dir)
print("wandb on :", wbu.wandb_enabled(cfg["train"].get("wandb_project")))

### run it

In [ ]:
t0 = time.time()
meta = train(cfg, seed=1, out_dir=out_dir, smoke=True, cfg_hash=h)
print(f"\nwall clock: {(time.time() - t0) / 60:.1f} min")
meta

### what it left behind

In [ ]:
for f in sorted(os.listdir(out_dir)):
    p = os.path.join(out_dir, f)
    print(f"{os.path.getsize(p) / 1e6:9.2f} MB  {f}")

with open(os.path.join(out_dir, "train_meta.json")) as fh:
    meta = json.load(fh)

# The full meta carries the per-site activation grids, which are long; print the
# fields that say whether the run was sane, and keep `meta` around to inspect.
for k in ("config", "config_hash", "variant", "weight", "act_bits", "params",
          "trainable_params", "smoke", "epochs_run", "n_files", "n_train",
          "n_val", "best_epoch", "best_val_macro_auc", "train_seconds"):
    print(f"  {k:20s} {meta[k]}")
print("\nact sites tracked:", len(meta["act_grid_after"]),
      "| grids that moved:",
      sum(1 for k in meta["act_grid_before"]
          if meta["act_grid_before"][k] != meta["act_grid_after"][k]))

---
A `model_best.keras` plus a `train_meta.json` with `"smoke": true` means the
training path is intact end to end.

To change what is being tested, point `CONFIG` at a different file in
`configs/` — `r14-l1x3-n8-w1a4.json` (4-bit activations),
`...-fp32.json` (the float baseline), `...-n64-...` (64 constituents). The
architecture and precision live entirely in that JSON.